# 03 - Full thread and aggregation

The six-contribution thread from section 11.A, signed in publication order,
then the aggregation query. Local only.

In [ ]:
# Setup. Local only: signs with the local profile, never publishes.
import sys, os, subprocess, glob
try:
    import nanopub, rdflib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nanopub", "rdflib"])
sys.path.insert(0, os.getcwd())
try:
    import na_nanopub as na
except ModuleNotFoundError:
    hits = glob.glob(os.path.join(os.getcwd(), "**", "na_nanopub.py"), recursive=True)
    if hits:
        sys.path.insert(0, os.path.dirname(hits[0]))
    import na_nanopub as na
import nanopub as _np
print("nanopub", _np.__version__, "-- ready, signing locally (no network)")

## Build the thread

Each contribution references the trusty URI of something already signed.
The rebuttal targets the counter, not the root.

In [ ]:
root = na.make(
    '''  sub:claim a schema:Statement ;
    rdf:value             "Certain gut microbiota profiles have stronger mRNA vaccine responses." ;
    cito:citesAsEvidence  <https://doi.org/10.1234/kim-2025-microbiota-vaccine> .''',
    attributed_to="orcid:0000-0001-alvarez-b", nanopub_type="schema:Statement", introduces="sub:claim", name="03-root")
claim = root.source_uri + "/claim"

support = na.make(
    f'''  sub:reason a schema:Statement ;
    rdf:value "Their SCFA data is compelling." ;
    cito:supports <{claim}> .''',
    attributed_to="orcid:0000-0002-wang-p", nanopub_type="cito:supports", introduces="sub:reason", name="03-support")

caveat = na.make(
    f'''  sub:caveat a schema:Statement ;
    rdf:value "Be cautious about interpreting causality here without intervention studies." ;
    cito:qualifies <{claim}> .''',
    attributed_to="<https://example.org/agents/medai-bot>", nanopub_type="cito:qualifies", introduces="sub:caveat", name="03-caveat")

counter = na.make(
    f'''  sub:counterclaim a schema:Statement ;
    rdf:value "The correlation could be confounded by diet." ;
    cito:disputes <{claim}> .''',
    attributed_to="orcid:0000-0003-moller-m", nanopub_type="cito:disputes", introduces="sub:counterclaim", name="03-counter")
counterclaim = counter.source_uri + "/counterclaim"

rebuttal = na.make(
    f'''  sub:rebuttal a schema:Statement ;
    rdf:value             "Note that [Kim 2025] controlled for fiber intake using dietary logs." ;
    cito:citesAsEvidence  <https://doi.org/10.1234/kim-2025-microbiota-vaccine> ;
    as:inReplyTo          <{counterclaim}> ;
    cito:disputes         <{counterclaim}> .''',
    attributed_to="<https://example.org/agents/criticai-bot>", nanopub_type="cito:disputes", introduces="sub:rebuttal", name="03-rebuttal")

question = na.make(
    f'''  sub:question a schema:Question ;
    rdf:value     "Has anyone tested this in non-trial populations?" ;
    as:inReplyTo  <{claim}> .''',
    attributed_to="orcid:0000-0002-perez-l", nanopub_type="schema:Question", introduces="sub:question", name="03-question")

thread = [root, support, caveat, counter, rebuttal, question]
for np in thread:
    na.show(np)

## Group by relation, with cited evidence

In [ ]:
ds = na.load(*thread)
q = na.QP + f'''
SELECT ?relation ?contribution ?value ?evidence WHERE {{ GRAPH ?g {{
  ?contribution ?relation <{claim}> ;
                rdf:value ?value .
  OPTIONAL {{ ?contribution cito:citesAsEvidence ?evidence }}
  FILTER(?relation IN (cito:supports, cito:disputes, cito:extends,
                       cito:agreesWith, cito:qualifies, as:inReplyTo))
}} }} ORDER BY ?relation'''

buckets = {}
for r in ds.query(q):
    buckets.setdefault(na.localname(r.relation), []).append(
        (r.value, str(r.evidence) if r.evidence else None))

print(f"Claim: {claim}\n")
for rel, items in buckets.items():
    print(f"[{rel}]  ({len(items)})")
    for value, ev in items:
        print(f"   - {value}")
        if ev:
            print(f"       evidence: {ev}")

## Count by relation

In [ ]:
q = na.QP + f'''
SELECT ?relation (COUNT(?c) AS ?n) WHERE {{ GRAPH ?g {{
  ?c ?relation <{claim}> .
  FILTER(?relation IN (cito:supports, cito:disputes, cito:qualifies, as:inReplyTo))
}} }} GROUP BY ?relation ORDER BY DESC(?n)'''
for r in ds.query(q):
    print(f"  {na.localname(r.relation):12s} {r.n}")